# 11 Raw Weather Forecast Benchmark

## Purpose

This notebook constructs a direct raw weather-forecast benchmark for Hong Kong temperature-threshold markets.

The previous notebooks created a look-ahead-safe threshold dataset and a simple climatological baseline classifier. This notebook adds a second benchmark: a raw forecast benchmark based on lead-time-specific weather-model forecasts. The benchmark uses the forecast daily maximum temperature directly and converts it into deterministic threshold-exceedance signals.

The key point is that this is not a trained post-processing model. It is a transparent raw-forecast benchmark. It answers the question: if the raw weather forecast says the daily maximum temperature is above a threshold, how well would that simple forecast-based rule perform against the official settlement outcome?

The notebook uses the Open-Meteo Previous Runs API as the first practical route for lead-time-specific weather-model forecasts. This is a feasible proxy route for the raw AI/weather-forecast benchmark while direct AIFS/Earth-2 feature extraction remains under development.

## 1. Imports and paths

In [3]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo
from io import StringIO
import re

import requests
import pandas as pd
import numpy as np

RAW_DIR = Path("../data/raw/raw_weather_benchmark")
PROCESSED_DIR = Path("../data/processed/raw_weather_benchmark")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 120)

print("Notebook run time UTC:", datetime.now(timezone.utc).isoformat())
print("Raw directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)

Notebook run time UTC: 2026-06-17T05:14:20.726089+00:00
Raw directory: ../data/raw/raw_weather_benchmark
Processed directory: ../data/processed/raw_weather_benchmark


## 2. Configuration

The first benchmark uses the same Hong Kong market example as the threshold-classification dataset. The target is the official Hong Kong Observatory daily maximum temperature on the local settlement date.

In [6]:
CONFIG = {
    "city": "Hong Kong",
    "local_timezone": "Asia/Hong_Kong",
    "latitude": 22.3027,
    "longitude": 114.1740,
    "contract_slug": "highest-temperature-in-hong-kong-on-may-30-2026",
    "polymarket_event_url": "https://polymarket.com/event/highest-temperature-in-hong-kong-on-may-30-2026",
    "settlement_date_local": "2026-05-30",
    "temperature_unit": "celsius",
    "threshold_grid_fallback": list(range(24, 34)),
}

PREVIOUS_RUNS_BASE = "https://previous-runs-api.open-meteo.com/v1/forecast"
HKO_OPEN_DATA_URL = "https://data.weather.gov.hk/weatherAPI/opendata/opendata.php?dataType=CLMMAXT&rformat=csv&station=HKO"

CONFIG

{'city': 'Hong Kong',
 'local_timezone': 'Asia/Hong_Kong',
 'latitude': 22.3027,
 'longitude': 114.174,
 'contract_slug': 'highest-temperature-in-hong-kong-on-may-30-2026',
 'polymarket_event_url': 'https://polymarket.com/event/highest-temperature-in-hong-kong-on-may-30-2026',
 'settlement_date_local': '2026-05-30',
 'temperature_unit': 'celsius',
 'threshold_grid_fallback': [24, 25, 26, 27, 28, 29, 30, 31, 32, 33]}

## 3. Helper functions

In [9]:
def fetch_text(url, params=None, timeout=40):
    try:
        response = requests.get(url, params=params, timeout=timeout)
        return {
            "url": response.url,
            "status_code": response.status_code,
            "ok": response.ok,
            "content_type": response.headers.get("content-type"),
            "text": response.text,
            "error": None,
        }
    except Exception as e:
        return {
            "url": url,
            "status_code": None,
            "ok": False,
            "content_type": None,
            "text": "",
            "error": repr(e),
        }


def fetch_json(url, params=None, timeout=40):
    result = fetch_text(url, params=params, timeout=timeout)
    print("URL:", result["url"])
    print("Status:", result["status_code"])
    if not result["ok"]:
        raise RuntimeError(f"Request failed: {result['status_code']} {result['error']} {result['text'][:300]}")
    return requests.models.complexjson.loads(result["text"])


def normalise_col(col):
    return re.sub(r"[^a-z0-9]+", "_", str(col).strip().lower()).strip("_")


def read_hko_csv_flexibly(csv_text, max_skiprows=30):
    """Read an HKO CSV that may contain metadata rows before the real table."""
    lines = csv_text.splitlines()

    for i, line in enumerate(lines[:max_skiprows + 1]):
        normalised = line.lower().replace(" ", "")
        if "year" in normalised and "month" in normalised and "day" in normalised:
            return pd.read_csv(StringIO(csv_text), skiprows=i), i

    for skiprows in range(max_skiprows + 1):
        try:
            df = pd.read_csv(StringIO(csv_text), skiprows=skiprows)
            cols = [normalise_col(c) for c in df.columns]
            has_ymd = {"year", "month", "day"}.issubset(set(cols))
            if has_ymd and df.shape[0] > 100:
                return df, skiprows
        except Exception:
            continue

    raise ValueError("Could not parse HKO CSV into a usable table.")


def standardise_hko_daily_max(df):
    out = df.copy()
    out.columns = [normalise_col(c) for c in out.columns]

    if {"year", "month", "day"}.issubset(out.columns):
        date_series = pd.to_datetime(
            {
                "year": pd.to_numeric(out["year"], errors="coerce"),
                "month": pd.to_numeric(out["month"], errors="coerce"),
                "day": pd.to_numeric(out["day"], errors="coerce"),
            },
            errors="coerce",
        )
    else:
        raise ValueError("Expected Year, Month and Day columns in HKO data.")

    temp_candidates = [
        c for c in out.columns
        if c in ["value", "temperature", "temp"]
        or ("max" in c and ("temp" in c or "temperature" in c))
        or ("maximum" in c and ("temp" in c or "temperature" in c))
    ]

    if temp_candidates:
        temp_col = temp_candidates[0]
    else:
        numeric_scores = {}
        for c in out.columns:
            if c in ["year", "month", "day"]:
                continue
            numeric_scores[c] = pd.to_numeric(out[c], errors="coerce").notna().sum()
        temp_col = max(numeric_scores, key=numeric_scores.get)

    clean = pd.DataFrame({
        "date": date_series,
        "official_daily_max_temperature_c": pd.to_numeric(out[temp_col], errors="coerce"),
    })

    clean = clean.dropna(subset=["date", "official_daily_max_temperature_c"])
    clean["date"] = pd.to_datetime(clean["date"])
    clean["date_str"] = clean["date"].dt.date.astype(str)
    clean["year"] = clean["date"].dt.year
    clean["month"] = clean["date"].dt.month
    clean["day"] = clean["date"].dt.day
    clean["source_route"] = "HKO open data CSV"

    return clean


def local_day_window_utc(local_date_str, timezone_name):
    local_tz = ZoneInfo(timezone_name)
    local_start = datetime.fromisoformat(local_date_str).replace(tzinfo=local_tz)
    local_end = local_start + timedelta(days=1) - timedelta(seconds=1)
    local_midpoint = local_start + timedelta(hours=12)

    return {
        "target_local_date": local_date_str,
        "target_local_day_start_utc": local_start.astimezone(timezone.utc),
        "target_local_day_midpoint_utc": local_midpoint.astimezone(timezone.utc),
        "target_local_day_end_utc": local_end.astimezone(timezone.utc),
    }


def brier_score(y_true, y_prob):
    y_true = pd.Series(y_true).astype(float)
    y_prob = pd.Series(y_prob).astype(float)
    mask = y_true.notna() & y_prob.notna()
    if mask.sum() == 0:
        return np.nan
    return float(np.mean((y_prob[mask] - y_true[mask]) ** 2))


def safe_log_score(y_true, y_prob, eps=1e-6):
    y_true = pd.Series(y_true).astype(float)
    y_prob = pd.Series(y_prob).astype(float).clip(eps, 1 - eps)
    mask = y_true.notna() & y_prob.notna()
    if mask.sum() == 0:
        return np.nan
    return float(-np.mean(y_true[mask] * np.log(y_prob[mask]) + (1 - y_true[mask]) * np.log(1 - y_prob[mask])))

## 4. Load official HKO settlement value

The benchmark target uses the official settlement-source value rather than the Open-Meteo realised proxy.

In [12]:
settlement_saved_path = Path("../data/processed/hko/hko_official_settlement_temperature.csv")

def load_or_retrieve_hko_settlement(target_date):
    if settlement_saved_path.exists():
        saved = pd.read_csv(settlement_saved_path)
        if "settlement_date_local" in saved.columns and "official_realised_temperature_c" in saved.columns:
            match = saved[
                (saved["settlement_date_local"].astype(str) == target_date)
                & saved["official_realised_temperature_c"].notna()
            ].copy()
            if not match.empty:
                row = match.iloc[0]
                return pd.DataFrame([{
                    "settlement_date_local": target_date,
                    "official_realised_temperature_c": float(row["official_realised_temperature_c"]),
                    "settlement_source": row.get("settlement_source", "Hong Kong Observatory"),
                    "source_route": row.get("source_route", "saved HKO settlement table"),
                    "retrieval_status": "loaded_saved_hko_settlement_table",
                }])

    result = fetch_text(HKO_OPEN_DATA_URL)
    print("HKO status:", result["status_code"], "chars:", len(result["text"]))
    if not result["ok"] or not result["text"].strip():
        raise RuntimeError("Could not retrieve HKO open data.")

    (RAW_DIR / "hko_daily_max_temperature.csv").write_text(result["text"], encoding="utf-8")
    raw, skiprows_used = read_hko_csv_flexibly(result["text"])
    print("HKO CSV parsed with skiprows:", skiprows_used)

    hko_daily = standardise_hko_daily_max(raw)
    match = hko_daily[hko_daily["date_str"] == target_date].copy()

    if match.empty:
        return pd.DataFrame([{
            "settlement_date_local": target_date,
            "official_realised_temperature_c": np.nan,
            "settlement_source": "Hong Kong Observatory",
            "source_route": "HKO open data CSV",
            "retrieval_status": "not_found",
        }])

    return pd.DataFrame([{
        "settlement_date_local": target_date,
        "official_realised_temperature_c": float(match.iloc[0]["official_daily_max_temperature_c"]),
        "settlement_source": "Hong Kong Observatory",
        "source_route": "HKO open data CSV",
        "retrieval_status": "matched_open_data_csv",
    }])

settlement_df = load_or_retrieve_hko_settlement(CONFIG["settlement_date_local"])
settlement_df

,settlement_date_local,official_realised_temperature_c,settlement_source,source_route,retrieval_status
0,2026-05-30,32.6,Hong Kong Observatory,HKO open data CSV,loaded_saved_hko_settlement_table


## 5. Load the Step 7 threshold dataset if available

If Notebook 9 has already been run locally, this notebook uses its threshold grid and market-implied threshold probabilities. If the file is not present, the notebook falls back to a simple Hong Kong threshold grid.

In [15]:
step7_threshold_path = Path("../data/processed/threshold_dataset/hong_kong_threshold_classification_dataset.csv")
step8_baseline_path = Path("../data/processed/baseline_classifier/hong_kong_step7_baseline_classifier_predictions.csv")

if step7_threshold_path.exists():
    step7_df = pd.read_csv(step7_threshold_path)
    print("Loaded Step 7 threshold dataset:", step7_threshold_path)
    print("Step 7 shape:", step7_df.shape)
else:
    step7_df = pd.DataFrame()
    print("Step 7 threshold dataset not found. Fallback threshold grid will be used.")

if step8_baseline_path.exists():
    step8_df = pd.read_csv(step8_baseline_path)
    print("Loaded Step 8 baseline-scored Step 7 dataset:", step8_baseline_path)
    print("Step 8 scored shape:", step8_df.shape)
else:
    step8_df = pd.DataFrame()
    print("Step 8 baseline predictions not found. Baseline comparison will be skipped.")

if not step7_df.empty and "threshold_c" in step7_df.columns:
    threshold_grid = sorted(step7_df["threshold_c"].dropna().astype(float).unique().tolist())
else:
    threshold_grid = [float(x) for x in CONFIG["threshold_grid_fallback"]]

print("Threshold grid:", threshold_grid)

display(step7_df.head() if not step7_df.empty else pd.DataFrame({"threshold_c": threshold_grid}))

Loaded Step 7 threshold dataset: ../data/processed/threshold_dataset/hong_kong_threshold_classification_dataset.csv
Step 7 shape: (40, 26)
Loaded Step 8 baseline-scored Step 7 dataset: ../data/processed/baseline_classifier/hong_kong_step7_baseline_classifier_predictions.csv
Step 8 scored shape: (40, 37)
Threshold grid: [24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0]


,contract_slug,city,target_local_date,settlement_source,settlement_variable,official_realised_temperature_c,threshold_c,realised_exceeds_threshold,forecast_model,forecast_run_time_utc,forecast_issue_time_utc,forecast_valid_time_utc,timing_label,forecast_timing_category,lead_time_to_valid_hours,lead_time_to_day_start_hours,lead_time_to_day_midpoint_hours,lead_time_to_day_end_hours,market_implied_threshold_prob_raw,market_implied_threshold_prob_normalised,number_of_contributing_bins,average_price_time_gap_minutes,market_price_timestamp_rule,forecast_temperature_c,forecast_temperature_source,forecast_feature_status
0,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,24.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9645,0.998964,10,0.1,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge
1,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,25.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9620,0.996375,9,0.1,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge
2,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,26.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9595,0.993786,8,0.1,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge
3,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,27.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9540,0.988089,7,0.1,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge
4,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,28.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9505,0.984464,6,0.1,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge


## 6. Retrieve Open-Meteo Previous Runs forecasts

The Previous Runs API provides lead-time-specific hourly temperature variables. For this benchmark, each variable is converted into a daily maximum temperature over the local target date.

The variables are interpreted as raw forecast lead-time views. The resulting benchmark is deliberately simple and deterministic: for each threshold, the raw forecast probability is one if the forecast daily maximum is above the threshold and zero otherwise.

In [18]:
previous_run_vars = ["temperature_2m"] + [f"temperature_2m_previous_day{i}" for i in range(1, 8)]

params = {
    "latitude": CONFIG["latitude"],
    "longitude": CONFIG["longitude"],
    "start_date": CONFIG["settlement_date_local"],
    "end_date": CONFIG["settlement_date_local"],
    "hourly": ",".join(previous_run_vars),
    "timezone": CONFIG["local_timezone"],
    "temperature_unit": CONFIG["temperature_unit"],
}

previous_runs_json = fetch_json(PREVIOUS_RUNS_BASE, params=params)
(RAW_DIR / "open_meteo_previous_runs_hong_kong.json").write_text(
    pd.Series(previous_runs_json).to_json(indent=2),
    encoding="utf-8",
)

hourly = pd.DataFrame(previous_runs_json.get("hourly", {}))
hourly["time"] = pd.to_datetime(hourly["time"], errors="coerce")

display(hourly.head())
display(hourly.tail())

URL: https://previous-runs-api.open-meteo.com/v1/forecast?latitude=22.3027&longitude=114.174&start_date=2026-05-30&end_date=2026-05-30&hourly=temperature_2m%2Ctemperature_2m_previous_day1%2Ctemperature_2m_previous_day2%2Ctemperature_2m_previous_day3%2Ctemperature_2m_previous_day4%2Ctemperature_2m_previous_day5%2Ctemperature_2m_previous_day6%2Ctemperature_2m_previous_day7&timezone=Asia%2FHong_Kong&temperature_unit=celsius
Status: 200


,time,temperature_2m,temperature_2m_previous_day1,temperature_2m_previous_day2,temperature_2m_previous_day3,temperature_2m_previous_day4,temperature_2m_previous_day5,temperature_2m_previous_day6,temperature_2m_previous_day7
0,2026-05-30 00:00:00,24.9,26.6,27.4,27.0,27.0,27.0,26.2,26.7
1,2026-05-30 01:00:00,25.1,26.2,26.9,26.9,26.8,27.1,26.0,26.5
2,2026-05-30 02:00:00,25.3,25.4,26.5,26.8,26.5,27.0,25.8,26.2
3,2026-05-30 03:00:00,25.2,25.4,26.2,26.5,26.0,26.5,25.5,26.0
4,2026-05-30 04:00:00,25.2,25.5,25.4,25.8,25.6,25.9,25.1,25.7


,time,temperature_2m,temperature_2m_previous_day1,temperature_2m_previous_day2,temperature_2m_previous_day3,temperature_2m_previous_day4,temperature_2m_previous_day5,temperature_2m_previous_day6,temperature_2m_previous_day7
19,2026-05-30 19:00:00,27.5,27.6,26.7,25.9,26.4,26.5,24.9,26.3
20,2026-05-30 20:00:00,28.0,26.8,25.5,26.4,26.0,26.2,25.9,23.6
21,2026-05-30 21:00:00,27.5,26.5,25.6,26.2,25.8,26.1,25.8,23.6
22,2026-05-30 22:00:00,27.4,26.2,25.8,26.1,25.9,26.1,25.8,23.9
23,2026-05-30 23:00:00,27.4,26.1,25.9,25.9,26.0,26.2,25.8,24.1


In [20]:
def extract_previous_run_lead_days(variable_name):
    if variable_name == "temperature_2m":
        return 0
    match = re.search(r"previous_day(\d+)", variable_name)
    if match:
        return int(match.group(1))
    return np.nan

raw_forecast_rows = []

for variable in previous_run_vars:
    if variable not in hourly.columns:
        continue

    values = pd.to_numeric(hourly[variable], errors="coerce")
    if values.notna().sum() == 0:
        continue

    raw_forecast_rows.append({
        "city": CONFIG["city"],
        "target_local_date": CONFIG["settlement_date_local"],
        "forecast_source": "Open-Meteo Previous Runs API",
        "forecast_variable": variable,
        "lead_time_days_previous_run": extract_previous_run_lead_days(variable),
        "raw_forecast_daily_max_c": float(values.max()),
        "raw_forecast_daily_min_c": float(values.min()),
        "raw_forecast_daily_mean_c": float(values.mean()),
        "hourly_observations_used": int(values.notna().sum()),
        "temperature_unit": "deg C",
    })

raw_forecast_df = pd.DataFrame(raw_forecast_rows).sort_values("lead_time_days_previous_run").reset_index(drop=True)

raw_forecast_df["official_realised_temperature_c"] = settlement_df["official_realised_temperature_c"].iloc[0]
raw_forecast_df["forecast_error_c"] = raw_forecast_df["raw_forecast_daily_max_c"] - raw_forecast_df["official_realised_temperature_c"]

raw_forecast_df

,city,target_local_date,forecast_source,forecast_variable,lead_time_days_previous_run,raw_forecast_daily_max_c,raw_forecast_daily_min_c,raw_forecast_daily_mean_c,hourly_observations_used,temperature_unit,official_realised_temperature_c,forecast_error_c
0,Hong Kong,2026-05-30,Open-Meteo Previous Runs API,temperature_2m,0,31.2,24.9,27.725000,24,deg C,32.6,-1.4
1,Hong Kong,2026-05-30,Open-Meteo Previous Runs API,temperature_2m_previous_day1,1,29.6,25.0,27.154167,24,deg C,32.6,-3.0
2,Hong Kong,2026-05-30,Open-Meteo Previous Runs API,temperature_2m_previous_day2,2,30.5,24.0,26.858333,24,deg C,32.6,-2.1
3,Hong Kong,2026-05-30,Open-Meteo Previous Runs API,temperature_2m_previous_day3,3,29.9,23.5,26.937500,24,deg C,32.6,-2.7
4,Hong Kong,2026-05-30,Open-Meteo Previous Runs API,temperature_2m_previous_day4,4,30.0,24.1,26.970833,24,deg C,32.6,-2.6
5,Hong Kong,2026-05-30,Open-Meteo Previous Runs API,temperature_2m_previous_day5,5,29.5,24.8,26.962500,24,deg C,32.6,-3.1
6,Hong Kong,2026-05-30,Open-Meteo Previous Runs API,temperature_2m_previous_day6,6,26.2,24.0,24.983333,24,deg C,32.6,-6.4
7,Hong Kong,2026-05-30,Open-Meteo Previous Runs API,temperature_2m_previous_day7,7,27.4,23.6,25.804167,24,deg C,32.6,-5.2


## 7. Build the raw threshold benchmark

For each forecast lead time and threshold, the raw point forecast is converted into a deterministic threshold probability.

A deterministic raw forecast benchmark is intentionally simple:

```text
raw_forecast_probability = 1 if raw_forecast_daily_max_c >= threshold_c, otherwise 0
```

This is not a calibrated probability model, but it is a transparent raw weather-forecast benchmark.

In [23]:
benchmark_rows = []
realised_temperature = settlement_df["official_realised_temperature_c"].iloc[0]

for _, forecast_row in raw_forecast_df.iterrows():
    for threshold_c in threshold_grid:
        realised_exceeds_threshold = int(realised_temperature >= threshold_c) if pd.notna(realised_temperature) else np.nan
        raw_prob = float(forecast_row["raw_forecast_daily_max_c"] >= threshold_c)

        benchmark_rows.append({
            "city": CONFIG["city"],
            "contract_slug": CONFIG["contract_slug"],
            "target_local_date": CONFIG["settlement_date_local"],
            "settlement_source": settlement_df["settlement_source"].iloc[0],
            "official_realised_temperature_c": realised_temperature,
            "threshold_c": float(threshold_c),
            "realised_exceeds_threshold": realised_exceeds_threshold,
            "forecast_source": forecast_row["forecast_source"],
            "forecast_variable": forecast_row["forecast_variable"],
            "lead_time_days_previous_run": forecast_row["lead_time_days_previous_run"],
            "raw_forecast_daily_max_c": forecast_row["raw_forecast_daily_max_c"],
            "raw_forecast_error_c": forecast_row["forecast_error_c"],
            "raw_forecast_margin_to_threshold_c": forecast_row["raw_forecast_daily_max_c"] - float(threshold_c),
            "raw_forecast_threshold_probability": raw_prob,
            "raw_forecast_brier_component": (raw_prob - realised_exceeds_threshold) ** 2 if pd.notna(realised_exceeds_threshold) else np.nan,
            "benchmark_type": "deterministic_raw_point_forecast",
        })

raw_threshold_benchmark_df = pd.DataFrame(benchmark_rows)

print("Raw threshold benchmark shape:", raw_threshold_benchmark_df.shape)
display(raw_threshold_benchmark_df.head(40))

Raw threshold benchmark shape: (80, 16)


,city,contract_slug,target_local_date,settlement_source,official_realised_temperature_c,threshold_c,realised_exceeds_threshold,forecast_source,forecast_variable,lead_time_days_previous_run,raw_forecast_daily_max_c,raw_forecast_error_c,raw_forecast_margin_to_threshold_c,raw_forecast_threshold_probability,raw_forecast_brier_component,benchmark_type
0,Hong Kong,highest-temperature-in-hong-kong-on-may-30-2026,2026-05-30,Hong Kong Observatory,32.6,24.0,1,Open-Meteo Previous Runs API,temperature_2m,0,31.2,-1.4,7.2,1.0,0.0,deterministic_raw_point_forecast
1,Hong Kong,highest-temperature-in-hong-kong-on-may-30-2026,2026-05-30,Hong Kong Observatory,32.6,25.0,1,Open-Meteo Previous Runs API,temperature_2m,0,31.2,-1.4,6.2,1.0,0.0,deterministic_raw_point_forecast
2,Hong Kong,highest-temperature-in-hong-kong-on-may-30-2026,2026-05-30,Hong Kong Observatory,32.6,26.0,1,Open-Meteo Previous Runs API,temperature_2m,0,31.2,-1.4,5.2,1.0,0.0,deterministic_raw_point_forecast
3,Hong Kong,highest-temperature-in-hong-kong-on-may-30-2026,2026-05-30,Hong Kong Observatory,32.6,27.0,1,Open-Meteo Previous Runs API,temperature_2m,0,31.2,-1.4,4.2,1.0,0.0,deterministic_raw_point_forecast
4,Hong Kong,highest-temperature-in-hong-kong-on-may-30-2026,2026-05-30,Hong Kong Observatory,32.6,28.0,1,Open-Meteo Previous Runs API,temperature_2m,0,31.2,-1.4,3.2,1.0,0.0,deterministic_raw_point_forecast
5,Hong Kong,highest-temperature-in-hong-kong-on-may-30-2026,2026-05-30,Hong Kong Observatory,32.6,29.0,1,Open-Meteo Previous Runs API,temperature_2m,0,31.2,-1.4,2.2,1.0,0.0,deterministic_raw_point_forecast
6,Hong Kong,highest-temperature-in-hong-kong-on-may-30-2026,2026-05-30,Hong Kong Observatory,32.6,30.0,1,Open-Meteo Previous Runs API,temperature_2m,0,31.2,-1.4,1.2,1.0,0.0,deterministic_raw_point_forecast
7,Hong Kong,highest-temperature-in-hong-kong-on-may-30-2026,2026-05-30,Hong Kong Observatory,32.6,31.0,1,Open-Meteo Previous Runs API,temperature_2m,0,31.2,-1.4,0.2,1.0,0.0,deterministic_raw_point_forecast
8,Hong Kong,highest-temperature-in-hong-kong-on-may-30-2026,2026-05-30,Hong Kong Observatory,32.6,32.0,1,Open-Meteo Previous Runs API,temperature_2m,0,31.2,-1.4,-0.8,0.0,1.0,deterministic_raw_point_forecast
9,Hong Kong,highest-temperature-in-hong-kong-on-may-30-2026,2026-05-30,Hong Kong Observatory,32.6,33.0,0,Open-Meteo Previous Runs API,temperature_2m,0,31.2,-1.4,-1.8,0.0,0.0,deterministic_raw_point_forecast


## 8. Raw benchmark diagnostics

In [26]:
raw_metrics_by_lead = (
    raw_threshold_benchmark_df
    .groupby("lead_time_days_previous_run")
    .agg(
        rows=("raw_forecast_threshold_probability", "count"),
        forecast_daily_max_c=("raw_forecast_daily_max_c", "first"),
        forecast_error_c=("raw_forecast_error_c", "first"),
        brier_score=("raw_forecast_brier_component", "mean"),
        accuracy=("raw_forecast_brier_component", lambda x: float((x == 0).mean())),
    )
    .reset_index()
)

print("Raw benchmark metrics by previous-run lead day:")
display(raw_metrics_by_lead)

raw_overall_metrics = pd.DataFrame([{
    "model_or_benchmark": "raw_previous_runs_deterministic_threshold",
    "brier_score": brier_score(
        raw_threshold_benchmark_df["realised_exceeds_threshold"],
        raw_threshold_benchmark_df["raw_forecast_threshold_probability"],
    ),
    "log_score": safe_log_score(
        raw_threshold_benchmark_df["realised_exceeds_threshold"],
        raw_threshold_benchmark_df["raw_forecast_threshold_probability"],
    ),
    "rows": len(raw_threshold_benchmark_df),
}])

raw_overall_metrics

Raw benchmark metrics by previous-run lead day:


,lead_time_days_previous_run,rows,forecast_daily_max_c,forecast_error_c,brier_score,accuracy
0,0,10,31.2,-1.4,0.1,0.9
1,1,10,29.6,-3.0,0.3,0.7
2,2,10,30.5,-2.1,0.2,0.8
3,3,10,29.9,-2.7,0.3,0.7
4,4,10,30.0,-2.6,0.2,0.8
5,5,10,29.5,-3.1,0.3,0.7
6,6,10,26.2,-6.4,0.6,0.4
7,7,10,27.4,-5.2,0.5,0.5


,model_or_benchmark,brier_score,log_score,rows
0,raw_previous_runs_deterministic_threshold,0.3125,4.317348,80


## 9. Compare with Step 7 market probabilities and Step 8 baseline if available

The Step 7 timing table uses exact forecast issue times, while Open-Meteo Previous Runs provides lead-day forecast variables. This section maps each Step 7 row to the nearest previous-run lead day using the lead time to the local-day midpoint.

This comparison is an approximation, but it is useful for a first benchmark table. The timing caveat is kept explicit in the output.

In [29]:
def map_step7_lead_to_previous_run_days(lead_time_hours):
    if pd.isna(lead_time_hours):
        return np.nan
    return int(np.clip(np.round(float(lead_time_hours) / 24), 0, 7))

if step7_df.empty:
    step7_raw_comparison_df = pd.DataFrame()
    print("Step 7 threshold dataset not available; market comparison skipped.")
else:
    step7_compare = step7_df.copy()
    step7_compare["lead_time_days_previous_run"] = step7_compare["lead_time_to_day_midpoint_hours"].apply(
        map_step7_lead_to_previous_run_days
    )
    step7_compare["threshold_c"] = step7_compare["threshold_c"].astype(float)

    join_cols = ["lead_time_days_previous_run", "threshold_c"]
    raw_join = raw_threshold_benchmark_df[
        join_cols + [
            "raw_forecast_daily_max_c",
            "raw_forecast_error_c",
            "raw_forecast_margin_to_threshold_c",
            "raw_forecast_threshold_probability",
            "benchmark_type",
        ]
    ].copy()

    step7_raw_comparison_df = step7_compare.merge(raw_join, on=join_cols, how="left")

    if not step8_df.empty:
        step8_keep_cols = [
            c for c in [
                "contract_slug",
                "target_local_date",
                "timing_label",
                "threshold_c",
                "baseline_classifier_probability",
                "baseline_model_name",
            ]
            if c in step8_df.columns
        ]
        step8_join = step8_df[step8_keep_cols].drop_duplicates()
        step7_raw_comparison_df = step7_raw_comparison_df.merge(
            step8_join,
            on=["contract_slug", "target_local_date", "timing_label", "threshold_c"],
            how="left",
        )

    step7_raw_comparison_df["raw_forecast_brier_component"] = (
        step7_raw_comparison_df["raw_forecast_threshold_probability"]
        - step7_raw_comparison_df["realised_exceeds_threshold"]
    ) ** 2

    if "market_implied_threshold_prob_normalised" in step7_raw_comparison_df.columns:
        step7_raw_comparison_df["market_brier_component"] = (
            step7_raw_comparison_df["market_implied_threshold_prob_normalised"]
            - step7_raw_comparison_df["realised_exceeds_threshold"]
        ) ** 2

    if "baseline_classifier_probability" in step7_raw_comparison_df.columns:
        step7_raw_comparison_df["baseline_brier_component"] = (
            step7_raw_comparison_df["baseline_classifier_probability"]
            - step7_raw_comparison_df["realised_exceeds_threshold"]
        ) ** 2

    print("Step 7 raw benchmark comparison shape:", step7_raw_comparison_df.shape)
    display(step7_raw_comparison_df.head(40))

Step 7 raw benchmark comparison shape: (40, 37)


,contract_slug,city,target_local_date,settlement_source,settlement_variable,official_realised_temperature_c,threshold_c,realised_exceeds_threshold,forecast_model,forecast_run_time_utc,forecast_issue_time_utc,forecast_valid_time_utc,timing_label,forecast_timing_category,lead_time_to_valid_hours,lead_time_to_day_start_hours,lead_time_to_day_midpoint_hours,lead_time_to_day_end_hours,market_implied_threshold_prob_raw,market_implied_threshold_prob_normalised,number_of_contributing_bins,average_price_time_gap_minutes,market_price_timestamp_rule,forecast_temperature_c,forecast_temperature_source,forecast_feature_status,lead_time_days_previous_run,raw_forecast_daily_max_c,raw_forecast_error_c,raw_forecast_margin_to_threshold_c,raw_forecast_threshold_probability,benchmark_type,baseline_classifier_probability,baseline_model_name,raw_forecast_brier_component,market_brier_component,baseline_brier_component
0,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,24.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9645,0.998964,10,0.100000,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge,2,30.5,-2.1,6.5,1.0,deterministic_raw_point_forecast,0.992736,logistic_regression_calendar_threshold_baseline,0.0,1.072742e-06,0.000053
1,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,25.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9620,0.996375,9,0.100000,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge,2,30.5,-2.1,5.5,1.0,deterministic_raw_point_forecast,0.984324,logistic_regression_calendar_threshold_baseline,0.0,1.314109e-05,0.000246
2,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,26.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9595,0.993786,8,0.100000,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge,2,30.5,-2.1,4.5,1.0,deterministic_raw_point_forecast,0.966503,logistic_regression_calendar_threshold_baseline,0.0,3.861873e-05,0.001122
3,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,27.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9540,0.988089,7,0.100000,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge,2,30.5,-2.1,3.5,1.0,deterministic_raw_point_forecast,0.929862,logistic_regression_calendar_threshold_baseline,0.0,1.418702e-04,0.004919
4,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,28.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9505,0.984464,6,0.100000,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge,2,30.5,-2.1,2.5,1.0,deterministic_raw_point_forecast,0.858992,logistic_regression_calendar_threshold_baseline,0.0,2.413670e-04,0.019883
5,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,29.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+0

In [31]:
comparison_metric_rows = []

if not step7_raw_comparison_df.empty:
    comparison_metric_rows.append({
        "model_or_benchmark": "raw_previous_runs_deterministic_threshold",
        "brier_score": brier_score(
            step7_raw_comparison_df["realised_exceeds_threshold"],
            step7_raw_comparison_df["raw_forecast_threshold_probability"],
        ),
        "log_score": safe_log_score(
            step7_raw_comparison_df["realised_exceeds_threshold"],
            step7_raw_comparison_df["raw_forecast_threshold_probability"],
        ),
        "rows": len(step7_raw_comparison_df),
    })

    if "market_implied_threshold_prob_normalised" in step7_raw_comparison_df.columns:
        comparison_metric_rows.append({
            "model_or_benchmark": "market_implied_threshold_probability",
            "brier_score": brier_score(
                step7_raw_comparison_df["realised_exceeds_threshold"],
                step7_raw_comparison_df["market_implied_threshold_prob_normalised"],
            ),
            "log_score": safe_log_score(
                step7_raw_comparison_df["realised_exceeds_threshold"],
                step7_raw_comparison_df["market_implied_threshold_prob_normalised"],
            ),
            "rows": len(step7_raw_comparison_df),
        })

    if "baseline_classifier_probability" in step7_raw_comparison_df.columns:
        comparison_metric_rows.append({
            "model_or_benchmark": "calendar_threshold_baseline_classifier",
            "brier_score": brier_score(
                step7_raw_comparison_df["realised_exceeds_threshold"],
                step7_raw_comparison_df["baseline_classifier_probability"],
            ),
            "log_score": safe_log_score(
                step7_raw_comparison_df["realised_exceeds_threshold"],
                step7_raw_comparison_df["baseline_classifier_probability"],
            ),
            "rows": len(step7_raw_comparison_df),
        })

comparison_metrics_df = pd.DataFrame(comparison_metric_rows)
comparison_metrics_df

,model_or_benchmark,brier_score,log_score,rows
0,raw_previous_runs_deterministic_threshold,0.225000,3.108491,40
1,market_implied_threshold_probability,0.052922,0.162932,40
2,calendar_threshold_baseline_classifier,0.131270,0.381431,40


## 10. Save outputs

In [34]:
raw_forecast_output_path = PROCESSED_DIR / "hong_kong_raw_previous_runs_daily_forecasts.csv"
raw_threshold_output_path = PROCESSED_DIR / "hong_kong_raw_weather_threshold_benchmark.csv"
raw_metrics_output_path = PROCESSED_DIR / "hong_kong_raw_weather_benchmark_metrics.csv"
comparison_output_path = PROCESSED_DIR / "hong_kong_step7_raw_weather_benchmark_comparison.csv"
comparison_metrics_output_path = PROCESSED_DIR / "hong_kong_step7_raw_weather_benchmark_comparison_metrics.csv"

raw_forecast_df.to_csv(raw_forecast_output_path, index=False)
raw_threshold_benchmark_df.to_csv(raw_threshold_output_path, index=False)
raw_metrics_by_lead.to_csv(raw_metrics_output_path, index=False)

if not step7_raw_comparison_df.empty:
    step7_raw_comparison_df.to_csv(comparison_output_path, index=False)
    comparison_metrics_df.to_csv(comparison_metrics_output_path, index=False)

print("Saved raw forecast table:", raw_forecast_output_path)
print("Saved raw threshold benchmark:", raw_threshold_output_path)
print("Saved raw benchmark metrics:", raw_metrics_output_path)

if not step7_raw_comparison_df.empty:
    print("Saved Step 7 comparison table:", comparison_output_path)
    print("Saved Step 7 comparison metrics:", comparison_metrics_output_path)
else:
    print("No Step 7 comparison outputs saved because Step 7 data was unavailable.")

Saved raw forecast table: ../data/processed/raw_weather_benchmark/hong_kong_raw_previous_runs_daily_forecasts.csv
Saved raw threshold benchmark: ../data/processed/raw_weather_benchmark/hong_kong_raw_weather_threshold_benchmark.csv
Saved raw benchmark metrics: ../data/processed/raw_weather_benchmark/hong_kong_raw_weather_benchmark_metrics.csv
Saved Step 7 comparison table: ../data/processed/raw_weather_benchmark/hong_kong_step7_raw_weather_benchmark_comparison.csv
Saved Step 7 comparison metrics: ../data/processed/raw_weather_benchmark/hong_kong_step7_raw_weather_benchmark_comparison_metrics.csv


## 11. Interpretation

This notebook adds a direct raw weather-forecast benchmark to the project.

The benchmark uses lead-time-specific Open-Meteo Previous Runs forecasts as a feasible first raw weather-model source. For each lead time, the hourly forecast path over the Hong Kong settlement date is converted into a forecast daily maximum temperature. For each threshold, this raw point forecast is then mapped into a deterministic threshold signal.

This is deliberately not a trained probability model. It is a transparent raw-forecast benchmark against which later supervised post-processing models can be compared. It also avoids the earlier arbitrary Gaussian residual conversion because it does not claim to produce a continuous predictive distribution.

The benchmark is useful for three reasons. First, it tests whether raw forecast information alone can predict the realised threshold outcome. Second, it provides a comparison against market-implied probabilities and the calendar-threshold baseline classifier where the Step 7 and Step 8 outputs are available. Third, it prepares the pipeline for later replacement or extension with direct AIFS, ensemble or Earth-2 forecast features.

The main caveat is that the Previous Runs lead-day variables are not exactly the same as a precisely timestamped AIFS or Earth-2 model run. Therefore, the lead-day comparison should be treated as a practical raw forecast benchmark rather than the final timestamp-perfect AI weather forecast benchmark. Later notebooks should attach model-specific forecast run times and valid times more explicitly.